In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [ ]:
OUTPUT_DIR = Path("stage5_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE2_DIR = Path("stage2_factor_outputs")
STAGE3_DIR = Path("stage3_model_outputs")
STAGE4_DIR = Path("stage4_financial_outputs")

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_read_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

factor_speed = safe_read_csv(STAGE2_DIR / "factor_speed_bin_table.csv")
factor_delivery = safe_read_csv(STAGE2_DIR / "factor_delivery_service_table.csv")
factor_manager = safe_read_csv(STAGE2_DIR / "factor_manager_table.csv")
factor_region = safe_read_csv(STAGE2_DIR / "factor_region_table.csv")
factor_rf = safe_read_csv(STAGE2_DIR / "factor_rf_importance.csv")
factor_logit = safe_read_csv(STAGE2_DIR / "factor_logistic_coefficients.csv")

model_metrics = safe_read_csv(STAGE3_DIR / "model_metrics.csv")
score_bins = safe_read_csv(STAGE3_DIR / "score_bin_performance.csv")
predictions = safe_read_csv(STAGE3_DIR / "buyout_predictions.csv")

model_business = {}
if (STAGE3_DIR / "model_business_summary.json").exists():
    model_business = load_json(STAGE3_DIR / "model_business_summary.json")

loss_region = safe_read_csv(STAGE4_DIR / "loss_by_region.csv")
loss_delivery = safe_read_csv(STAGE4_DIR / "loss_by_delivery.csv")
loss_manager = safe_read_csv(STAGE4_DIR / "loss_by_manager.csv")
loss_reason = safe_read_csv(STAGE4_DIR / "loss_by_rejection_reason.csv")

overall_financial = load_json(STAGE4_DIR / "financial_overall_summary.json")
speed_impact = load_json(STAGE4_DIR / "financial_speed_impact.json")
risk_summary = load_json(STAGE4_DIR / "financial_risk_summary.json")
financial_business = load_json(STAGE4_DIR / "financial_business_summary.json")

In [ ]:
print("factor_speed:", factor_speed.shape)
print("factor_delivery:", factor_delivery.shape)
print("factor_manager:", factor_manager.shape)
print("factor_region:", factor_region.shape)
print("factor_rf:", factor_rf.shape)
print("factor_logit:", factor_logit.shape)

print("model_metrics:", model_metrics.shape)
print("score_bins:", score_bins.shape)
print("predictions:", predictions.shape)

print("loss_region:", loss_region.shape)
print("loss_delivery:", loss_delivery.shape)
print("loss_manager:", loss_manager.shape)
print("loss_reason:", loss_reason.shape)

In [ ]:
insights = {}

if not model_metrics.empty:
    best_model_row = model_metrics.sort_values(["roc_auc", "pr_auc"], ascending=False).iloc[0]
    insights["best_model"] = {
        "model": best_model_row["model"],
        "roc_auc": float(best_model_row["roc_auc"]),
        "pr_auc": float(best_model_row["pr_auc"]),
        "accuracy": float(best_model_row["accuracy"]),
        "precision": float(best_model_row["precision"]) if "precision" in best_model_row else None,
        "recall": float(best_model_row["recall"]) if "recall" in best_model_row else None,
        "f1": float(best_model_row["f1"]) if "f1" in best_model_row else None,
    }
else:
    insights["best_model"] = None

if not factor_rf.empty:
    top_rf = factor_rf.iloc[0]
    insights["top_rf_feature"] = {
        "feature": top_rf["feature"],
        "importance": float(top_rf["importance"])
    }
else:
    insights["top_rf_feature"] = None

if not factor_logit.empty:
    factor_logit = factor_logit.copy()
    coef_col = "coef" if "coef" in factor_logit.columns else factor_logit.columns[1]
    factor_logit["abs_coef"] = factor_logit[coef_col].abs()
    top_logit = factor_logit.sort_values("abs_coef", ascending=False).iloc[0]
    insights["top_logit_feature"] = {
        "feature": top_logit["feature"],
        "coef": float(top_logit[coef_col])
    }
else:
    insights["top_logit_feature"] = None

insights["top_region_loss"] = loss_region.iloc[0].to_dict() if not loss_region.empty else None
insights["top_delivery_loss"] = loss_delivery.iloc[0].to_dict() if not loss_delivery.empty else None
insights["top_manager_loss"] = loss_manager.iloc[0].to_dict() if not loss_manager.empty else None

insights["speed_impact"] = speed_impact

insights["risk_summary"] = risk_summary

with open(OUTPUT_DIR / "stage5_key_insights.json", "w", encoding="utf-8") as f:
    json.dump(insights, f, ensure_ascii=False, indent=2)

insights

In [ ]:
def recommendation_row(
    action,
    problem_area,
    evidence,
    estimated_gain_orders,
    estimated_gain_money,
    implementation_effort,
    priority_comment
):
    return {
        "action": action,
        "problem_area": problem_area,
        "evidence": evidence,
        "estimated_gain_orders": estimated_gain_orders,
        "estimated_gain_money": estimated_gain_money,
        "implementation_effort": implementation_effort,
        "priority_comment": priority_comment,
    }

recommendations = []

In [ ]:
if speed_impact.get("potential_saved_revenue") is not None:
    recommendations.append(
        recommendation_row(
            action="Сократить время передачи заказа в доставку",
            problem_area="B2C / операционный процесс",
            evidence=(
                f"При speed_to_delivery_days <= {speed_impact['threshold_days']} "
                f"выкуп = {speed_impact['fast_buyout_rate']:.3f}, "
                f"при > {speed_impact['threshold_days']} = {speed_impact['slow_buyout_rate']:.3f}"
            ),
            estimated_gain_orders=speed_impact["potential_saved_orders"],
            estimated_gain_money=speed_impact["potential_saved_revenue"],
            implementation_effort="Средний",
            priority_comment="Самый сильный рычаг: факторно значим и даёт прямой денежный эффект"
        )
    )

In [ ]:
if not loss_delivery.empty:
    top_delivery = loss_delivery.iloc[0]
    recommendations.append(
        recommendation_row(
            action=f"Провести аудит и оптимизацию delivery service: {top_delivery['delivery_service']}",
            problem_area="Delivery / логистика",
            evidence=(
                f"Максимальные потери по доставке: {top_delivery['delivery_service']}, "
                f"orders={int(top_delivery['orders'])}, "
                f"buyout_rate={top_delivery['buyout_rate']:.3f}, "
                f"total_loss={top_delivery['total_loss']:.2f}"
            ),
            estimated_gain_orders=top_delivery["non_buyouts"] * 0.15,
            estimated_gain_money=top_delivery["total_loss"] * 0.15,
            implementation_effort="Средний",
            priority_comment="Хороший кандидат для пилота: сегмент большой и потери высокие"
        )
    )

In [ ]:
if not loss_manager.empty:
    top_manager = loss_manager.iloc[0]
    recommendations.append(
        recommendation_row(
            action=f"Запустить аудит и обучение для менеджера/группы: {top_manager['manager_id']}",
            problem_area="B2C / менеджеры",
            evidence=(
                f"Максимальные потери по менеджеру: {top_manager['manager_id']}, "
                f"orders={int(top_manager['orders'])}, "
                f"buyout_rate={top_manager['buyout_rate']:.3f}, "
                f"total_loss={top_manager['total_loss']:.2f}"
            ),
            estimated_gain_orders=top_manager["non_buyouts"] * 0.10,
            estimated_gain_money=top_manager["total_loss"] * 0.10,
            implementation_effort="Низкий",
            priority_comment="Быстрое внедрение: управленческая мера без изменения платформы"
        )
    )

In [ ]:
if not loss_region.empty:
    top_region = loss_region.iloc[0]
    recommendations.append(
        recommendation_row(
            action=f"Проверить логистику/сегментацию в регионе: {top_region['region']}",
            problem_area="Региональная стратегия",
            evidence=(
                f"Максимальные потери по региону: {top_region['region']}, "
                f"orders={int(top_region['orders'])}, "
                f"buyout_rate={top_region['buyout_rate']:.3f}, "
                f"total_loss={top_region['total_loss']:.2f}"
            ),
            estimated_gain_orders=top_region["non_buyouts"] * 0.08,
            estimated_gain_money=top_region["total_loss"] * 0.08,
            implementation_effort="Средний",
            priority_comment="Стоит запускать после проверки качества заполнения поля region"
        )
    )

In [ ]:
if risk_summary and risk_summary.get("high_risk_orders") is not None:
    high_risk_loss = risk_summary.get("high_risk_total_loss", 0)
    recommendations.append(
        recommendation_row(
            action="Внедрить score-модель для выделения high-risk заказов",
            problem_area="ML / операционное управление",
            evidence=(
                f"High-risk сегмент (< {risk_summary['risk_threshold']}) = "
                f"{risk_summary['high_risk_orders']} заказов, "
                f"buyout_rate={risk_summary['high_risk_buyout_rate']:.3f}, "
                f"loss={high_risk_loss:.2f}"
            ),
            estimated_gain_orders=risk_summary["high_risk_orders"] * 0.05,
            estimated_gain_money=high_risk_loss * 0.12,
            implementation_effort="Средний",
            priority_comment="Даёт приоритизацию: можно адресно обрабатывать проблемные заказы"
        )
    )

In [ ]:
if not loss_reason.empty:
    recommendations.append(
        recommendation_row(
            action="Улучшить качество учёта причин отказов",
            problem_area="Data Quality",
            evidence=(
                "Поле rejection_category используется для анализа потерь, "
                "но без качественной разметки причины отказов теряется глубина управленческого анализа"
            ),
            estimated_gain_orders=np.nan,
            estimated_gain_money=np.nan,
            implementation_effort="Низкий",
            priority_comment="Не даёт мгновенного финансового эффекта, но критично для точной диагностики"
        )
    )

In [ ]:
rec_df = pd.DataFrame(recommendations)
rec_df

In [ ]:
effort_map = {
    "Низкий": 1,
    "Средний": 2,
    "Высокий": 3
}
rec_df["effort_score"] = rec_df["implementation_effort"].map(effort_map)

rec_df["estimated_gain_money_filled"] = rec_df["estimated_gain_money"].fillna(0)

max_gain = rec_df["estimated_gain_money_filled"].max() if len(rec_df) else 1
if max_gain == 0:
    max_gain = 1

rec_df["impact_score"] = rec_df["estimated_gain_money_filled"] / max_gain
rec_df["priority_score"] = rec_df["impact_score"] / rec_df["effort_score"]

rec_df = rec_df.sort_values(
    ["priority_score", "estimated_gain_money_filled"],
    ascending=[False, False]
).reset_index(drop=True)

rec_df.to_csv(OUTPUT_DIR / "final_recommendations.csv", index=False)
rec_df

In [ ]:
final_summary = {
    "total_estimated_loss": overall_financial["total_estimated_loss"],
    "buyout_rate": overall_financial["buyout_rate"],
    "best_model": insights["best_model"]["model"] if insights["best_model"] else None,
    "best_model_roc_auc": insights["best_model"]["roc_auc"] if insights["best_model"] else None,
    "top_priority_action": rec_df.iloc[0]["action"] if len(rec_df) else None,
    "top_priority_expected_gain_money": float(rec_df.iloc[0]["estimated_gain_money"]) if len(rec_df) and pd.notna(rec_df.iloc[0]["estimated_gain_money"]) else None,
    "top_priority_comment": rec_df.iloc[0]["priority_comment"] if len(rec_df) else None,
    "recommendations_count": int(len(rec_df))
}

with open(OUTPUT_DIR / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, ensure_ascii=False, indent=2)

print(json.dumps(final_summary, ensure_ascii=False, indent=2))

In [ ]:
if len(rec_df):
    plt.figure(figsize=(12, 7))
    plt.barh(rec_df["action"], rec_df["priority_score"])
    plt.xlabel("Priority score")
    plt.title("Приоритизация управленческих рекомендаций")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "recommendations_priority.png", dpi=200, bbox_inches="tight")
    plt.show()